# 02 · Data Transformation
*EV capacity planning for PowerCharge Utilities · notebook 2 of 8 · Anna Tafri*

Reads: `data/raw/*.csv`, decisions in `reports/data_quality_log.csv`  
Writes: `data/processed/ev_events_clean.csv`, `substations_clean.csv`, `ev_model_features.csv`, `reports/data_dictionary.csv`

### Goal
Turn three messy raw files into one analysis-ready table, with every row being one real charging event enriched with its weather and the substation that serves it. Each step below applies one decision from the Notebook 01 data-quality log, so a reviewer can trace every change back to its evidence.

| Step | Applies | What happens |
|---|---|---|
| 1 | DQ-01 | Remove duplicate EV rows |
| 2 | DQ-04 | Parse coordinate text into numeric lat/lon |
| 3 | DQ-02 | Resolve the duplicated `Substation_11` key |
| 4 | - | Standardise column names |
| 5 | - | Join EV + weather + substations |
| 6 | DQ-03 | Feature engineering (time, load, weather, distance) |
| 7 | - | Encoding and normalisation for modelling |
| 8 | - | Validation checks, then save |


## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Shared helpers live in src/ so later notebooks reuse the exact same logic
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from ev_utils import parse_coords, haversine_km, to_snake

ev_raw = pd.read_csv(RAW / "ev_distribution_dataset.csv", parse_dates=["Timestamp"])
geo_raw = pd.read_csv(RAW / "geospatial_dataset.csv")
wx_raw = pd.read_csv(RAW / "weather_dataset.csv", parse_dates=["Timestamp"])

# A running log of row counts, so the effect of every step is visible at the end
lineage = [("raw EV rows", len(ev_raw))]
print(ev_raw.shape, geo_raw.shape, wx_raw.shape)

(500, 10) (50, 3) (97, 5)


## 1. Remove duplicate EV rows (DQ-01)
The rows are identical in every column, so they are copies of the same event rather than separate events that look alike.

In [2]:
ev = ev_raw.drop_duplicates().reset_index(drop=True)
lineage.append(("after deduplication", len(ev)))
print(f"Removed {len(ev_raw) - len(ev)} duplicate rows -> {len(ev)} unique charging events")

# Sanity check: no two remaining rows share the same timestamp and station
assert not ev.duplicated(["Timestamp", "EV_Charging_Station_Location"]).any()

Removed 403 duplicate rows -> 97 unique charging events


## 2. Parse coordinates (DQ-04)

In [3]:
ev[["station_lat", "station_lon"]] = parse_coords(ev["EV_Charging_Station_Location"]).values

geo = geo_raw.copy()
geo[["sub_lat", "sub_lon"]] = parse_coords(geo["Substation_Location"]).values

ev[["Timestamp", "station_lat", "station_lon"]].head()

,Timestamp,station_lat,station_lon
0,2020-01-10 18:00:00,64.02,-129.53
1,2020-01-22 17:00:00,48.45,-71.99
2,2020-01-31 17:00:00,39.69,-79.06
3,2020-02-03 22:00:00,71.13,-112.20
4,2020-01-17 10:00:00,31.72,-83.49


## 3. Resolve the duplicated substation key (DQ-02)
`Substation_11` has two rows in the substation table, roughly 4,000 km apart, with capacities of 80 MW and 43 MW. A plain join would attach both substations to each Substation_11 event, which double counts that demand.

**Approach:** treat them as two physical assets (`Substation_11_A`, `Substation_11_B`). Then assign each of the two Substation_11 charging events to the nearer asset, because a charging station is fed from the nearby network.

In [4]:
dup_id = "Substation_11"
dup_mask = geo["Substation_ID"] == dup_id
geo.loc[dup_mask, "Substation_ID"] = [f"{dup_id}_{s}" for s in "AB"[: dup_mask.sum()]]
geo.loc[geo["Substation_ID"].str.startswith(dup_id)]

,Substation_ID,Substation_Location,Transmission_Line_Capacity (MW),sub_lat,sub_lon
19,Substation_11_A,"(66.83643347824741, -69.3563579228404)",80,66.84,-69.36
46,Substation_11_B,"(33.0822685351715, -113.11893127920547)",43,33.08,-113.12


In [5]:
candidates = geo.loc[geo["Substation_ID"].str.startswith(dup_id)]
ev_dup = ev["Substation_ID"] == dup_id

def nearest_candidate(row):
    d = haversine_km(row["station_lat"], row["station_lon"],
                     candidates["sub_lat"].values, candidates["sub_lon"].values)
    return candidates["Substation_ID"].iloc[int(np.argmin(d))]

# Keep the original ID for the weather join (weather still uses it)
ev["substation_id_raw"] = ev["Substation_ID"]
ev.loc[ev_dup, "Substation_ID"] = ev.loc[ev_dup].apply(nearest_candidate, axis=1)
ev.loc[ev_dup, ["Timestamp", "substation_id_raw", "Substation_ID", "station_lat", "station_lon"]]

,Timestamp,substation_id_raw,Substation_ID,station_lat,station_lon
38,2020-01-14 07:00:00,Substation_11,Substation_11_B,31.58,-112.34
39,2020-02-08 12:00:00,Substation_11,Substation_11_B,41.18,-92.73


In [6]:
assert geo["Substation_ID"].is_unique, "Substation_ID must now be a valid primary key"
print("Substation_ID is unique:", geo["Substation_ID"].is_unique, f"({len(geo)} substations)")

Substation_ID is unique: True (50 substations)


## 4. Standardise column names
Snake_case names without units in brackets are easier to type and safer in code. Units stay in the name as a suffix (`_kwh`, `_mw`, `_c`, `_mm`).

In [7]:
ev = ev.rename(columns=to_snake).rename(columns={
    "ev_charging_station_specifications": "charger_type",
    "ev_charging_station_location": "station_location_raw",
})
geo = geo.rename(columns=to_snake).rename(columns={
    "transmission_line_capacity_mw": "capacity_mw",
    "substation_location": "substation_location_raw",
})
wx = wx_raw.rename(columns=to_snake).rename(columns={"substation_id": "substation_id_raw"})

for name, df in {"ev": ev, "geo": geo, "weather": wx}.items():
    print(f"{name:<8}", list(df.columns))

ev       ['timestamp', 'geographical_area', 'customer_type', 'electricity_consumption_kwh', 'station_location_raw', 'charger_type', 'ev_type', 'charging_habit', 'number_of_evs', 'substation_id', 'station_lat', 'station_lon', 'substation_id_raw']
geo      ['substation_id', 'substation_location_raw', 'capacity_mw', 'sub_lat', 'sub_lon']
weather  ['timestamp', 'temperature_c', 'precipitation_mm', 'weather_conditions', 'substation_id_raw']


## 5. Join into one analysis table
- EV + weather on `timestamp + substation_id_raw`. Notebook 01 showed a 1:1 match.
- + substations on the resolved `substation_id`.

`validate=` makes pandas raise an error if a join is ever not one-to-one. That protects the row count against silent duplication if the data changes.

In [8]:
df = (
    ev.merge(wx, on=["timestamp", "substation_id_raw"], how="left", validate="one_to_one")
      .merge(geo, on="substation_id", how="left", validate="many_to_one")
)
lineage.append(("after joins", len(df)))

print(f"{len(df)} rows | missing weather: {df['temperature_c'].isna().sum()} "
      f"| missing substation: {df['capacity_mw'].isna().sum()}")

97 rows | missing weather: 0 | missing substation: 0


## 6. Feature engineering
Features are grouped by the question they help answer later.

### 6a. Time features: *when* does load happen?
The peak window is set to 17:00–21:00. That is the typical evening residential peak, when people get home and plug in, and it is when EV charging adds the most stress to the grid.

In [9]:
PEAK_HOURS = range(17, 21)  # 17:00-20:59

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["is_weekend"] = df["timestamp"].dt.dayofweek >= 5
df["is_peak_hour"] = df["hour"].isin(PEAK_HOURS)
df["time_of_day"] = pd.cut(
    df["hour"], bins=[-1, 5, 11, 16, 20, 23],
    labels=["Night (0-5)", "Morning (6-11)", "Afternoon (12-16)", "Evening peak (17-20)", "Late evening (21-23)"],
)
df[["timestamp", "hour", "day_of_week", "is_weekend", "is_peak_hour", "time_of_day"]].head()

,timestamp,hour,day_of_week,is_weekend,is_peak_hour,time_of_day
0,2020-01-10 18:00:00,18,Friday,False,True,Evening peak (17-20)
1,2020-01-22 17:00:00,17,Wednesday,False,True,Evening peak (17-20)
2,2020-01-31 17:00:00,17,Friday,False,True,Evening peak (17-20)
3,2020-02-03 22:00:00,22,Monday,False,False,Late evening (21-23)
4,2020-01-17 10:00:00,10,Friday,False,False,Morning (6-11)


### 6b. Load features: *how much* demand?
- `kwh_per_ev` normalises consumption by fleet size, so a site with 9 EVs is not automatically labelled "heavy".
- `nominal_charger_kw` is an assumption based on typical industry ratings. It is used later to convert energy (kWh) into power (kW / MW). Notebook 04 will stress-test it.

| Charger type | Assumed rating | Basis |
|---|---|---|
| Type 1 | 7.4 kW | Single-phase AC (J1772) |
| Type 2 | 22 kW | Three-phase AC (Mennekes) |
| DC Fast | 50 kW | Entry-level DC fast charger |

In [10]:
CHARGER_KW = {"Type 1": 7.4, "Type 2": 22.0, "DC Fast": 50.0}

df["kwh_per_ev"] = df["electricity_consumption_kwh"] / df["number_of_evs"]
df["nominal_charger_kw"] = df["charger_type"].map(CHARGER_KW)
assert df["nominal_charger_kw"].notna().all(), "Unmapped charger type found"

df[["electricity_consumption_kwh", "number_of_evs", "kwh_per_ev", "charger_type", "nominal_charger_kw"]].describe()

,electricity_consumption_kwh,number_of_evs,kwh_per_ev,nominal_charger_kw
count,97.00,97.00,97.00,97.00
mean,512.15,5.25,153.54,28.93
std,86.45,2.59,142.45,17.22
min,315.91,1.00,44.21,7.40
25%,441.06,3.00,63.93,22.00
50%,511.35,5.00,104.48,22.00
75%,563.39,8.00,152.01,50.00
max,718.98,9.00,689.68,50.00


### 6c. Weather features (DQ-03)
The condition label is unreliable, so the features come from temperature and precipitation. We keep the original label and add a flag for readings where it contradicts the temperature, so nothing is hidden.

In [11]:
df["temp_band"] = pd.cut(df["temperature_c"], bins=[-50, 15, 25, 30, 60],
                         labels=["Cool (<15)", "Mild (15-25)", "Warm (25-30)", "Hot (>30)"])
df["is_wet"] = df["precipitation_mm"] >= 2.5   # moderate rain threshold
df["weather_label_reliable"] = ~((df["weather_conditions"] == "Snowy") & (df["temperature_c"] > 2))

print(df["temp_band"].value_counts().sort_index().to_string(), "\n")
print(f"Records with an unreliable weather label: {(~df['weather_label_reliable']).sum()}")

temp_band
Cool (<15)       3
Mild (15-25)    41
Warm (25-30)    37
Hot (>30)       16 

Records with an unreliable weather label: 22


### 6d. Network feature: *how far* is each station from its substation?
Long feeders lose more voltage, so distance is a reliability signal for the bottleneck analysis.

In [12]:
df["station_to_sub_km"] = haversine_km(df["station_lat"], df["station_lon"], df["sub_lat"], df["sub_lon"])
df["station_to_sub_km"].describe()

count      97.00
mean    2,953.05
std     1,529.39
min       182.41
25%     1,935.92
50%     2,728.02
75%     3,606.45
max     7,117.81
Name: station_to_sub_km, dtype: float64

> **Note on distances:** the median station is about 2,700 km from its substation. In a real network, feeders are usually under 20 km. This is another sign that the coordinates are synthetic (DQ-05), so distance will be used as a relative ranking signal only.

## 7. Encoding and normalisation (modelling table)
The clean table keeps human-readable labels for EDA and reporting. A separate modelling table holds the machine-ready version:
- Ordinal encoding for `charging_habit`, which has a natural order of frequency: Occasional < Weekly < Daily.
- One-hot encoding for nominal categories with no order: area, customer type, EV type, charger type.
- Min-max scaling (0–1) on numeric drivers, so they can be combined into a risk score in Notebook 05 without one unit dominating.

In [13]:
HABIT_ORDER = {"Occasional": 1, "Weekly": 2, "Daily": 3}
NOMINAL = ["geographical_area", "customer_type", "ev_type", "charger_type"]
NUMERIC = ["electricity_consumption_kwh", "number_of_evs", "kwh_per_ev", "nominal_charger_kw",
           "temperature_c", "precipitation_mm", "capacity_mw", "station_to_sub_km"]

model = df[["timestamp", "substation_id", "hour", "is_weekend", "is_peak_hour"] + NOMINAL + NUMERIC].copy()
model["charging_habit_ord"] = df["charging_habit"].map(HABIT_ORDER)
model = pd.get_dummies(model, columns=NOMINAL, prefix=["area", "cust", "ev", "charger"], dtype=int)

for col in NUMERIC:
    lo, hi = model[col].min(), model[col].max()
    model[f"{col}_scaled"] = (model[col] - lo) / (hi - lo)

bool_cols = model.select_dtypes("bool").columns
model[bool_cols] = model[bool_cols].astype(int)
print(model.shape)
model.head()

(97, 34)


,timestamp,substation_id,hour,is_weekend,is_peak_hour,electricity_consumption_kwh,number_of_evs,kwh_per_ev,nominal_charger_kw,temperature_c,precipitation_mm,capacity_mw,station_to_sub_km,charging_habit_ord,area_Rural,area_Suburban,area_Urban,cust_Commercial,cust_Industrial,cust_Residential,ev_Electric Bike,ev_Electric Car,ev_Electric Scooter,charger_DC Fast,charger_Type 1,charger_Type 2,electricity_consumption_kwh_scaled,number_of_evs_scaled,kwh_per_ev_scaled,nominal_charger_kw_scaled,temperature_c_scaled,precipitation_mm_scaled,capacity_mw_scaled,station_to_sub_km_scaled
0,2020-01-10 18:00:00,Substation_907,18,0,1,714.39,8,89.30,22.00,17.73,2.94,40,"3,160.39",1,0,0,1,0,1,0,0,1,0,0,0,1,0.99,0.88,0.07,0.34,0.24,0.59,0.36,0.43
1,2020-01-22 17:00:00,Substation_907,17,0,1,554.34,5,110.87,22.00,31.15,4.03,40,982.32,2,1,0,0,0,0,1,0,0,1,0,0,1,0.59,0.50,0.10,0.34,0.86,0.81,0.36,0.12
2,2020-01-31 17:00:00,Substation_218,17,0,1,598.27,9,66.47,22.00,26.17,4.94,71,"3,387.93",2,0,1,0,1,0,0,0,0,1,0,0,1,0.70,1.00,0.03,0.34,0.63,0.99,0.73,0.46
3,2020-02-03 22:00:00,Substation_218,22,0,0,603.18,2,301.59,22.00,22.51,4.36,71,"1,026.81",3,1,0,0,0,1,0,0,0,1,0,0,1,0.71,0.12,0.40,0.34,0.46,0.88,0.73,0.12
4,2020-01-17 10:00:00,Substation_339,10,0,0,617.94,5,123.59,7.40,18.71,2.44,85,"5,423.54",3,1,0,0,0,0,1,0,0,1,0,1,0,0.75,0.50,0.12,0.00,0.29,0.49,0.89,0.76


## 8. Validation checks
Automated checks act as a contract: if the raw data changes and breaks an assumption, the notebook fails loudly instead of producing wrong numbers quietly.

In [14]:
checks = {
    "97 unique events kept":            len(df) == 97,
    "no duplicate events":              not df.duplicated(["timestamp", "station_location_raw"]).any(),
    "every event has weather":          df["temperature_c"].notna().all(),
    "every event has a substation":     df["capacity_mw"].notna().all(),
    "substation key unique":            geo["substation_id"].is_unique,
    "consumption positive":             (df["electricity_consumption_kwh"] > 0).all(),
    "capacity positive":                (df["capacity_mw"] > 0).all(),
    "latitudes valid (-90..90)":        df[["station_lat", "sub_lat"]].abs().le(90).all().all(),
    "scaled features within 0..1":      model.filter(like="_scaled").apply(lambda c: c.between(0, 1)).all().all(),
    "modelling table has no nulls":     model.notna().all().all(),
}
result = pd.Series(checks, name="passed")
assert result.all(), result[~result]
result.to_frame()

,passed
97 unique events kept,True
no duplicate events,True
every event has weather,True
every event has a substation,True
substation key unique,True
consumption positive,True
capacity positive,True
latitudes valid (-90..90),True
scaled features within 0..1,True
modelling table has no nulls,True


In [15]:
pd.DataFrame(lineage, columns=["step", "rows"])

,step,rows
0,raw EV rows,500
1,after deduplication,97
2,after joins,97


## 9. Save outputs and data dictionary

In [16]:
CLEAN_COLS = [
    "timestamp", "date", "hour", "day_of_week", "is_weekend", "is_peak_hour", "time_of_day",
    "geographical_area", "customer_type", "ev_type", "charger_type", "charging_habit",
    "number_of_evs", "electricity_consumption_kwh", "kwh_per_ev", "nominal_charger_kw",
    "station_lat", "station_lon", "substation_id", "substation_id_raw", "sub_lat", "sub_lon",
    "capacity_mw", "station_to_sub_km",
    "temperature_c", "precipitation_mm", "weather_conditions", "weather_label_reliable", "temp_band", "is_wet",
]
ev_clean = df[CLEAN_COLS].sort_values("timestamp").reset_index(drop=True)
subs_clean = geo[["substation_id", "sub_lat", "sub_lon", "capacity_mw"]]

ev_clean.to_csv(PROCESSED / "ev_events_clean.csv", index=False)
subs_clean.to_csv(PROCESSED / "substations_clean.csv", index=False)
model.to_csv(PROCESSED / "ev_model_features.csv", index=False)

for f in sorted(PROCESSED.glob("*.csv")):
    print(f"saved {f.name:<26} {pd.read_csv(f).shape}")

saved bottleneck_scores.csv      (50, 20)
saved ev_events_clean.csv        (97, 30)
saved ev_model_features.csv      (97, 34)
saved substation_capacity.csv    (50, 17)


saved substations_clean.csv      (50, 4)


In [17]:
DESCRIPTIONS = {
    "timestamp": "Date and hour of the charging event",
    "date": "Calendar date of the event",
    "hour": "Hour of day (0-23)",
    "day_of_week": "Weekday name",
    "is_weekend": "True for Saturday/Sunday",
    "is_peak_hour": "True if hour is in the 17:00-20:59 evening peak",
    "time_of_day": "Five-band time-of-day category",
    "geographical_area": "Urban / Suburban / Rural",
    "customer_type": "Residential / Commercial / Industrial",
    "ev_type": "Electric Car / Scooter / Bike",
    "charger_type": "Type 1 / Type 2 / DC Fast",
    "charging_habit": "Daily / Weekly / Occasional",
    "number_of_evs": "Count of EVs at the data point",
    "electricity_consumption_kwh": "Energy consumed in the event (kWh)",
    "kwh_per_ev": "Consumption divided by number of EVs (kWh)",
    "nominal_charger_kw": "ASSUMED charger power rating by type (kW)",
    "station_lat": "Charging station latitude",
    "station_lon": "Charging station longitude",
    "substation_id": "Serving substation (Substation_11 split into _A/_B)",
    "substation_id_raw": "Substation ID as delivered in the raw data",
    "sub_lat": "Substation latitude",
    "sub_lon": "Substation longitude",
    "capacity_mw": "Transmission line capacity of the substation (MW)",
    "station_to_sub_km": "Great-circle distance station to substation (km)",
    "temperature_c": "Ambient temperature (°C)",
    "precipitation_mm": "Precipitation (mm)",
    "weather_conditions": "Weather label as delivered (low confidence, see DQ-03)",
    "weather_label_reliable": "False where label contradicts temperature",
    "temp_band": "Temperature band",
    "is_wet": "True if precipitation >= 2.5 mm",
}
data_dict = pd.DataFrame({
    "column": CLEAN_COLS,
    "dtype": [str(ev_clean[c].dtype) for c in CLEAN_COLS],
    "description": [DESCRIPTIONS[c] for c in CLEAN_COLS],
    "example": [ev_clean[c].iloc[0] for c in CLEAN_COLS],
})
data_dict.to_csv(REPORTS / "data_dictionary.csv", index=False)
data_dict

,column,dtype,description,example
0,timestamp,datetime64[us],Date and hour of the charging event,2020-01-03 11:00:00
1,date,object,Calendar date of the event,2020-01-03
2,hour,int32,Hour of day (0-23),11
3,day_of_week,str,Weekday name,Friday
4,is_weekend,bool,True for Saturday/Sunday,False
5,is_peak_hour,bool,True if hour is in the 17:00-20:59 evening peak,False
6,time_of_day,category,Five-band time-of-day category,Morning (6-11)
7,geographical_area,str,Urban / Suburban / Rural,Rural
8,customer_type,str,Residential / Commercial / Industrial,Residential
9,ev_type,str,Electric Car / Scooter / Bike,Electric Scooter


## 10. Summary and next step

**What changed**
- 500 raw rows became 97 unique charging events. Each one is joined to its weather reading and serving substation, with no missing values.
- The `Substation_11` conflict is resolved. The substation table now has a valid primary key across 50 assets.
- 12 engineered features cover time (peak flag, time of day), load (kWh per EV, charger rating), weather (temperature band, wet flag, label reliability), and network (station-to-substation distance).
- A separate modelling table holds one-hot, ordinal, and min-max-scaled versions of these features, ready for risk scoring.
- Ten automated validation checks guard all of the above.

Assumptions introduced (to revisit in Notebook 04)
- Evening peak window = 17:00–21:00
- Charger ratings: Type 1 = 7.4 kW, Type 2 = 22 kW, DC Fast = 50 kW

**Next:** `03_exploratory_analysis.ipynb`: when, where, and by whom is electricity consumed? How do area, customer type, EV and charger type, charging habit, and weather shape demand?